In [2]:
import pandas as pd
df = pd.read_parquet("genres_clean.parquet")

#création d'un dictionnaire avec les id des genres en clé et les noms des genres en valeur
genres_dict = dict(zip(df["genreId"], df["name"]))
print(genres_dict)


{16: 'Animation', 35: 'Comedy', 10751: 'Family', 12: 'Adventure', 14: 'Fantasy', 10749: 'Romance', 18: 'Drama', 28: 'Action', 80: 'Crime', 53: 'Thriller', 27: 'Horror', 36: 'History', 878: 'Science Fiction', 9648: 'Mystery', 10752: 'War', 10769: 'Foreign', 10402: 'Music', 99: 'Documentary', 37: 'Western', 10770: 'TV Movie'}


In [3]:
#transformation du csv en parquet
df = pd.read_csv("/home/a/projet_l2b/recomodo/scripts/dataset/movies_clean.csv")
df["releaseDate"] = pd.to_datetime(df["releaseDate"], errors="coerce")
df["voteCount"] = pd.to_numeric(df["voteCount"], errors="coerce")
df["voteAverage"] = pd.to_numeric(df["voteAverage"], errors="coerce")

df.to_parquet("/home/a/projet_l2b/recomodo/scripts/parquet/movies_clean.parquet", index=False)

In [11]:
movies = pd.read_parquet("movies_clean.parquet")
titles = movies['title'].tolist()
indices = pd.Series(movies.index, index=movies['title'])
#print(movies.head())
print(movies.dtypes)
print(titles)
print(indices)


movieId                 int64
title                  object
overview               object
genres                 object
keywords               object
releaseDate    datetime64[ns]
voteAverage           float64
voteCount             float64
posterPath             object
director               object
dtype: object
['Toy Story', 'Jumanji', 'Grumpier Old Men', 'Waiting to Exhale', 'Father of the Bride Part II', 'Heat', 'Sabrina', 'Tom and Huck', 'Sudden Death', 'GoldenEye', 'The American President', 'Dracula: Dead and Loving It', 'Balto', 'Nixon', 'Cutthroat Island', 'Casino', 'Sense and Sensibility', 'Four Rooms', 'Ace Ventura: When Nature Calls', 'Money Train', 'Get Shorty', 'Copycat', 'Assassins', 'Powder', 'Leaving Las Vegas', 'Othello', 'Now and Then', 'Persuasion', 'The City of Lost Children', 'Shanghai Triad', 'Dangerous Minds', 'Twelve Monkeys', 'Babe', 'Carrington', 'Dead Man Walking', 'It Takes Two', 'Clueless', 'Cry, the Beloved Country', 'Richard III', 'Dead Presidents', 'Resto

In [5]:
#récupération de la colonne genres, qui sont les id des genres associés à chaque film (liste de liste)
import ast
genres= movies["genres"].fillna('[]').apply(ast.literal_eval).to_list()
print(genres)

[[16, 35, 10751], [12, 14, 10751], [10749, 35], [35, 18, 10749], [35], [28, 80, 18, 53], [35, 10749], [28, 12, 18, 10751], [28, 12, 53], [12, 28, 53], [35, 18, 10749], [35, 27], [10751, 16, 12], [36, 18], [28, 12], [18, 80], [18, 10749], [80, 35], [80, 35, 12], [28, 35, 80], [35, 53, 80], [18, 53], [28, 12, 80, 53], [18, 14, 878, 53], [18, 10749], [18], [35, 18, 10751], [18, 10749], [14, 878, 12], [18, 80], [18, 80], [878, 53, 9648], [14, 18, 35, 10751], [36, 18, 10749], [18], [35, 10751, 10749], [35, 18, 10749], [18], [18, 10752], [28, 80, 18, 36], [18, 10749], [28, 14], [14, 18, 35, 53], [18, 10749], [80, 9648, 53], [12, 16, 18, 10751], [18, 10749], [18, 80, 53], [35, 10749], [18, 10769], [28, 12, 35, 10751], [18], [35, 18, 10749], [35, 18, 10749], [12, 10751, 14], [18, 53], [10402, 18, 10751], [35], [35, 10749], [35], [28, 878], [18, 10751], [35, 10749], [35], [27, 28, 53, 80], [28, 53, 10749], [35, 18, 10749], [18, 36], [18, 10749], [35, 10751], [27, 878], [18, 53], [18, 53], [1075

In [6]:
genres_n = []
for i in genres:
    temp = []
    for j in i:
        if j in genres_dict:
            temp.append(genres_dict[j])
    genres_n.append(' '.join(temp))
print(genres_n)


['Animation Comedy Family', 'Adventure Fantasy Family', 'Romance Comedy', 'Comedy Drama Romance', 'Comedy', 'Action Crime Drama Thriller', 'Comedy Romance', 'Action Adventure Drama Family', 'Action Adventure Thriller', 'Adventure Action Thriller', 'Comedy Drama Romance', 'Comedy Horror', 'Family Animation Adventure', 'History Drama', 'Action Adventure', 'Drama Crime', 'Drama Romance', 'Crime Comedy', 'Crime Comedy Adventure', 'Action Comedy Crime', 'Comedy Thriller Crime', 'Drama Thriller', 'Action Adventure Crime Thriller', 'Drama Fantasy Science Fiction Thriller', 'Drama Romance', 'Drama', 'Comedy Drama Family', 'Drama Romance', 'Fantasy Science Fiction Adventure', 'Drama Crime', 'Drama Crime', 'Science Fiction Thriller Mystery', 'Fantasy Drama Comedy Family', 'History Drama Romance', 'Drama', 'Comedy Family Romance', 'Comedy Drama Romance', 'Drama', 'Drama War', 'Action Crime Drama History', 'Drama Romance', 'Action Fantasy', 'Fantasy Drama Comedy Thriller', 'Drama Romance', 'Crime 

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(analyzer='word',ngram_range=(1, 2),min_df=1, stop_words='english')
tfidf_matrix = tf.fit_transform(genres_n)
print(tfidf_matrix.shape)

(31238, 396)


In [10]:
from sklearn.metrics.pairwise import linear_kernel
#cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
#cosine_sim[:4, :4]
idx = 0
scores = linear_kernel(tfidf_matrix[idx:idx+1], tfidf_matrix).flatten()
print(scores)

[1.         0.12437968 0.07402454 ... 0.         0.         0.        ]


In [15]:
def genre_recommendations(title):
    idx = indices[title]

    sim_scores = linear_kernel(tfidf_matrix[idx:idx+1], tfidf_matrix).flatten()
    sim_scores = list(enumerate(sim_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:21]
    movie_indices = [i[0] for i in sim_scores]
    return [titles[i] for i in movie_indices]

genre_recommendations("Toy Story")

['Oliver & Company',
 'The Wrong Trousers',
 'Meet the Deedles',
 'Toy Story 2',
 'Creature Comforts',
 'Chicken Run',
 'Monsters, Inc.',
 'The Looney, Looney, Looney Bugs Bunny Movie',
 'Looney Tunes: Back in Action',
 "Bon Voyage, Charlie Brown (and Don't Come Back!)",
 'Garfield',
 "Bébé's Kids",
 'The SpongeBob SquarePants Movie',
 'The Lion King 1½',
 'Hoodwinked!',
 'Garfield: A Tail of Two Kitties',
 'Barnyard',
 'A Boy Named Charlie Brown',
 'Meet the Robinsons',
 "Surf's Up"]